# RNN Classifier with LSTM Trained on Own Dataset (IMDB)

## General Settings

In [3]:
#!pip install datasets -q

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from datasets import load_dataset
from collections import Counter
import re
from IPython.display import clear_output
import matplotlib.pyplot as plt

# ==========================================
# HIPERPARAMETRY
# ==========================================
MAX_SEQ_LEN = 200      # Obcinamy lub wydłużamy każdą recenzję do 200 słów
MAX_VOCAB_SIZE = 15000 # Ile najpopularniejszych słów pamięta sieć
BATCH_SIZE = 64
EMBEDDING_DIM = 100    # Rozmiar "profilu" każdego słowa
HIDDEN_DIM = 128       # Pojemność warstwy LSTM
NUM_LAYERS = 2         # Głębokość LSTM
DROPOUT = 0.5          # Zapobiega przeuczeniu
EPOCHS = 5
LEARNING_RATE = 0.001

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Używane urządzenie: {DEVICE}")

Używane urządzenie: cpu


## Przygotowanie danych i słownika

In [6]:
print("Pobieranie bazy IMDB...")
# Używamy mniejszej próbki (5000 do treningu, 1000 do testów), żeby na zajęciach liczyło się to w minutę, a nie godzinę.
dataset = load_dataset("imdb")
train_data = dataset['train'].shuffle(seed=42).select(range(5000))
test_data = dataset['test'].shuffle(seed=42).select(range(1000))

# 1. Prosty Tokenizator (czyszczenie z HTML i znaków interpunkcyjnych)
def tokenize(text):
    text = re.sub(r'<[^>]+>', ' ', text) # Usuń tagi <br />
    text = re.sub(r'[^\w\s]', '', text)  # Usuń kropki, przecinki
    return text.lower().split()

# 2. Budowanie Słownika (Vocabulary)
print("Budowanie słownika...")
all_tokens = []
for item in train_data:
    all_tokens.extend(tokenize(item['text']))

# Wybieramy tylko MAX_VOCAB_SIZE najpopularniejszych słów
counter = Counter(all_tokens)
most_common = counter.most_common(MAX_VOCAB_SIZE)

# Tworzymy mapowanie słowo -> indeks. 
# Rezerwujemy indeks 0 dla PAD (wypełniacz pustych miejsc) i 1 dla UNK (słowo nieznane)
vocab = {"<PAD>": 0, "<UNK>": 1}
for word, _ in most_common:
    vocab[word] = len(vocab)

print(f"Słownik gotowy! Rozmiar: {len(vocab)} słów.")

# 3. Funkcja przygotowująca paczki (Batches) dla PyTorcha
def collate_fn(batch):
    labels = torch.tensor([item['label'] for item in batch], dtype=torch.long)
    texts = []
    
    for item in batch:
        tokens = tokenize(item['text'])
        # Zamiana słów na liczby (jeśli słowa nie ma w słowniku, dajemy 1 -> <UNK>)
        indices = [vocab.get(token, 1) for token in tokens[:MAX_SEQ_LEN]]
        
        # Jeśli recenzja jest krótsza niż 200 słów, dopisujemy zera (<PAD>)
        if len(indices) < MAX_SEQ_LEN:
            indices += [0] * (MAX_SEQ_LEN - len(indices))
            
        texts.append(indices)
        
    texts = torch.tensor(texts, dtype=torch.long)
    return texts, labels

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, collate_fn=collate_fn, shuffle=True)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, collate_fn=collate_fn)
print("Loadery gotowe!")

Pobieranie bazy IMDB...


Budowanie słownika...
Słownik gotowy! Rozmiar: 15002 słów.
Loadery gotowe!


## Definicja modelu LSTM (dwukierunkowy)

In [9]:
class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size, emb_dim, hid_dim, n_layers, dropout):
        super().__init__()
        
        # 1. Zamienia indeksy na wektory
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        
        # 2. Dwukierunkowe LSTM
        self.lstm = nn.LSTM(emb_dim, 
                            hid_dim, 
                            num_layers=n_layers, 
                            batch_first=True, 
                            bidirectional=True,
                            dropout=dropout if n_layers > 1 else 0)
        
        # 3. Zewnętrzny Dropout dla regularyzacji
        self.dropout = nn.Dropout(dropout)
        
        # 4. Klasyfikator (2 klasy: Negatywna = 0, Pozytywna = 1)
        # Ponieważ jest dwukierunkowy (bidirectional), rozmiar ukryty mnożymy x2
        self.fc = nn.Linear(hid_dim * 2, 2)
        
    def forward(self, x):
        # x kształt: [batch_size, max_seq_len] (np. [64, 200])
        embedded = self.dropout(self.embedding(x))
        
        # out: wszystkie stany ukryte (nie używamy)
        # hidden: ostatni stan ukryty (używamy)
        out, (hidden, cell) = self.lstm(embedded)
        
        # hidden kształt: [num_layers * num_directions, batch_size, hid_dim]
        # Pobieramy dwa ostatnie ukryte stany (jeden patrzący w przód, drugi w tył z ostatniej warstwy)
        hidden_forward = hidden[-2, :, :]
        hidden_backward = hidden[-1, :, :]
        
        # Sklejamy je ze sobą
        final_hidden = torch.cat((hidden_forward, hidden_backward), dim=1)
        
        # Przepuszczamy przez warstwę liniową
        logits = self.fc(self.dropout(final_hidden))
        return logits

model = SentimentLSTM(len(vocab), EMBEDDING_DIM, HIDDEN_DIM, NUM_LAYERS, DROPOUT).to(DEVICE)
print(model)

SentimentLSTM(
  (embedding): Embedding(15002, 100, padding_idx=0)
  (lstm): LSTM(100, 128, num_layers=2, batch_first=True, dropout=0.5, bidirectional=True)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc): Linear(in_features=256, out_features=2, bias=True)
)


## Pętla treningowa

In [12]:
# Jeśli nie masz tej biblioteki, pierwsza linijka ją zainstaluje
!pip install tqdm -q

In [35]:
import torch.optim as optim
from tqdm.auto import tqdm # Importujemy pasek postępu
from IPython.display import clear_output

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

train_losses = []
test_accuracies = []

print("Rozpoczynamy trenowanie z paskiem postępu...")

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss = 0
    
    # OWIJAMY NASZ TRAIN_LOADER W TQDM!
    # Dzięki temu zobaczymy pasek ładujący się od 0 do 78 (ilość paczek)
    progress_bar = tqdm(train_loader, desc=f'Epoka {epoch}/{EPOCHS} [Trening]')
    
    for texts, labels in progress_bar:
        texts, labels = texts.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        predictions = model(texts)
        loss = criterion(predictions, labels)
        
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        
        # Aktualizujemy pasek postępu, żeby pokazywał aktualny błąd
        progress_bar.set_postfix({'strata': f'{loss.item():.4f}'})
        
    # --- EWALUACJA (TEST) ---
    model.eval()
    correct = 0
    total = 0
    
    # Pasek postępu dla testowania
    test_bar = tqdm(test_loader, desc=f'Epoka {epoch}/{EPOCHS} [Testowanie]', leave=False)
    
    with torch.no_grad():
        for texts, labels in test_bar:
            texts, labels = texts.to(DEVICE), labels.to(DEVICE)
            predictions = model(texts)
            predicted_class = torch.argmax(predictions, dim=1)
            correct += (predicted_class == labels).sum().item()
            total += labels.size(0)
            
    avg_loss = epoch_loss / len(train_loader)
    accuracy = (correct / total) * 100
    
    train_losses.append(avg_loss)
    test_accuracies.append(accuracy)
    
    # Wyświetlamy podsumowanie pod spodem
    print(f"✅ Koniec Epoki {epoch} | Średnia Strata: {avg_loss:.4f} | Dokładność: {accuracy:.2f}%\n")

print("Koniec trenowania!")

Rozpoczynamy trenowanie z paskiem postępu...


Epoka 1/5 [Trening]:   0%|          | 0/79 [00:00<?, ?it/s]

Epoka 1/5 [Testowanie]:   0%|          | 0/16 [00:00<?, ?it/s]

✅ Koniec Epoki 1 | Średnia Strata: 0.6934 | Dokładność: 54.30%



Epoka 2/5 [Trening]:   0%|          | 0/79 [00:00<?, ?it/s]

...

Epoka 4/5 [Trening]:   0%|          | 0/79 [00:00<?, ?it/s]

✅ Koniec Epoki 4 | Średnia Strata: 0.6416 | Dokładność: 61.70%



Epoka 5/5 [Trening]:   0%|          | 0/79 [00:00<?, ?it/s]

Epoka 5/5 [Testowanie]:   0%|          | 0/16 [00:00<?, ?it/s]

✅ Koniec Epoki 5 | Średnia Strata: 0.6138 | Dokładność: 60.80%

Koniec trenowania!


## Testowanie

In [37]:
def predict_sentiment(model, review_text):
    model.eval()
    
    # Przetworzenie własnego tekstu tak samo jak danych treningowych
    tokens = tokenize(review_text)
    indices = [vocab.get(token, 1) for token in tokens[:MAX_SEQ_LEN]]
    if len(indices) < MAX_SEQ_LEN:
        indices += [0] * (MAX_SEQ_LEN - len(indices))
        
    # Zamiana na tensor i przesłanie na GPU/CPU
    tensor = torch.tensor(indices).unsqueeze(0).to(DEVICE)
    
    with torch.no_grad():
        prediction = model(tensor)
        # Softmax by dostać ładne procenty
        probabilities = torch.softmax(prediction, dim=1)[0]
        
    prob_neg = probabilities[0].item() * 100
    prob_pos = probabilities[1].item() * 100
    
    print(f"Recenzja: '{review_text}'")
    print(f"Wynik sieci:")
    print(f"Negatywna: {prob_neg:.2f}%")
    print(f"Pozytywna: {prob_pos:.2f}%")
    print("-" * 30)

# UWAGA: Używamy języka angielskiego, bo model uczył się na bazie IMDB!
predict_sentiment(model, "This movie was absolutely fantastic! The acting was brilliant.")
predict_sentiment(model, "Terrible waste of time. Do not watch this garbage.")
predict_sentiment(model, "It was okay, not great but not terrible either.")

Recenzja: 'This movie was absolutely fantastic! The acting was brilliant.'
Wynik sieci:
Negatywna: 83.12%
Pozytywna: 16.88%
------------------------------
Recenzja: 'Terrible waste of time. Do not watch this garbage.'
Wynik sieci:
Negatywna: 76.55%
Pozytywna: 23.45%
------------------------------
Recenzja: 'It was okay, not great but not terrible either.'
Wynik sieci:
Negatywna: 81.92%
Pozytywna: 18.08%
------------------------------


In [39]:
predict_sentiment(model, "Could be better.")

Recenzja: 'Could be better.'
Wynik sieci:
Negatywna: 75.21%
Pozytywna: 24.79%
------------------------------
